# Building Detection with GeoDeep

This notebook runs GeoDeep's pre-built `buildings` model on an aerial orthophoto and:
1. Runs detection and inspects outputs
2. Visualizes bounding boxes over the source imagery
3. Computes basic detection statistics (count, area, confidence distribution)

## References
- GeoDeep GitHub: https://github.com/uav4geo/GeoDeep

## 1. Setup

In [ ]:
import os
import urllib.request
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import geopandas as gpd
import rasterio
import geodeep

url = "https://github.com/uav4geo/GeoDeep/releases/download/v0.1.0/sample.tif"
image_path = "sample.tif"
if not os.path.exists(image_path):
    urllib.request.urlretrieve(url, image_path)

## 2. Run Building Detection

In [ ]:
geodeep.detect(
    input=image_path,
    output="buildings.geojson",
    model="buildings",
    confidence=0.4,
)

gdf = gpd.read_file("buildings.geojson")
print(f"Detected buildings: {len(gdf)}")
print(f"\nProperties available: {list(gdf.columns)}")
print(gdf.head())

## 3. Visualize with Confidence Colormap

In [ ]:
with rasterio.open(image_path) as src:
    img_array = src.read([1, 2, 3])
    img_array = np.moveaxis(img_array, 0, -1)
    extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(img_array, extent=extent)

# Color detections by confidence score
score_col = "score" if "score" in gdf.columns else gdf.columns[-1]
gdf.plot(
    ax=ax,
    column=score_col,
    cmap="RdYlGn",
    alpha=0.6,
    legend=True,
    legend_kwds={"label": "Confidence score"},
)
ax.set_title(f"Building detections (n={len(gdf)})")
plt.tight_layout()
plt.show()

## 4. Detection Statistics

In [ ]:
# Reproject to a projected CRS for area computation
gdf_proj = gdf.to_crs(gdf.estimate_utm_crs())
gdf_proj["area_m2"] = gdf_proj.geometry.area

print("Detection Statistics")
print("=" * 35)
print(f"Total detected buildings: {len(gdf_proj)}")
print(f"Mean area:    {gdf_proj['area_m2'].mean():.1f} m²")
print(f"Median area:  {gdf_proj['area_m2'].median():.1f} m²")
print(f"Max area:     {gdf_proj['area_m2'].max():.1f} m²")
print(f"Min area:     {gdf_proj['area_m2'].min():.1f} m²")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(gdf_proj["area_m2"], bins=20, edgecolor="black")
axes[0].set_xlabel("Building area (m²)")
axes[0].set_ylabel("Count")
axes[0].set_title("Building area distribution")

if score_col in gdf.columns:
    axes[1].hist(gdf[score_col], bins=20, edgecolor="black", color="green")
    axes[1].set_xlabel("Confidence score")
    axes[1].set_ylabel("Count")
    axes[1].set_title("Confidence score distribution")

plt.tight_layout()
plt.show()

## 5. Filter by Confidence and Area

In [ ]:
min_confidence = 0.6
min_area_m2 = 20

filtered = gdf_proj[
    (gdf_proj.get(score_col, 1.0) >= min_confidence) &
    (gdf_proj["area_m2"] >= min_area_m2)
]

print(f"After filtering (confidence>={min_confidence}, area>={min_area_m2}m²):")
print(f"  Remaining: {len(filtered)} of {len(gdf_proj)} detections")

# Save filtered detections
filtered.to_file("buildings_filtered.geojson", driver="GeoJSON")
print("Saved to buildings_filtered.geojson")